# 01 - Data Understanding

**Objectif :** charger le dataset initial, créer immédiatement un train/test brut, puis inspecter uniquement le train.

In [1]:
import pandas as pd
import plotly.express as px

from credit_risk_lab.config.settings import settings

print(f"Project root: {settings.project_root}")
print(f"Environment: {settings.environment}")

Project root: /Users/surelmanda/3-Mlops-Databricks-Projects/Clean-Architecture-MLops/Credit-Risk-Lab
Environment: development


## 1. Data source

In [2]:
from credit_risk_lab.infrastructure.data_sources import CsvLoanDataLoader

loader = CsvLoanDataLoader(
    path=settings.raw_data_path,
    sep=settings.raw_data_sep,
    encoding=settings.raw_data_encoding,
)

raw_df = loader.load()
raw_df.head()

2026-09-07 07:34:11 | INFO     | CSVDatasetRepository | credit_risk_lab.infrastructure.data_sources.csv_dataset_repository:load:51 - Chargement du fichier : /Users/surelmanda/3-Mlops-Databricks-Projects/Clean-Architecture-MLops/Credit-Risk-Lab/data/raw/loan_data.csv
2026-09-07 07:34:12 | INFO     | CSVDatasetRepository | credit_risk_lab.infrastructure.data_sources.csv_dataset_repository:load:59 - Dataset chargé (45000 lignes, 14 colonnes)


,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,female,Master,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,3.0,561,No,1
1,21.0,female,High School,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504,Yes,0
2,25.0,female,High School,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635,No,1
3,23.0,female,Bachelor,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675,No,1
4,24.0,male,Master,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586,No,1


## 2. Initial raw train/test split

In [3]:
from credit_risk_lab.application import DatasetSplitter, SplitConfig
from credit_risk_lab.infrastructure.data_sources import CSVDatasetRepository

initial_split_config = SplitConfig(
    test_size=0.10,
    random_state=settings.random_state,
    stratify=True,
)
initial_splitter = DatasetSplitter(initial_split_config)

raw_train_df, raw_test_df = initial_splitter.split(raw_df)
split_summary = initial_splitter.summary(raw_train_df, raw_test_df)

CSVDatasetRepository.save(raw_train_df, settings.raw_train_path)
CSVDatasetRepository.save(raw_test_df, settings.raw_test_path)

split_summary

,sample,rows,positive_rate
0,train,40500,0.222222
1,holdout,4500,0.222222


## 3. Initial leakage check

In [4]:
from credit_risk_lab.infrastructure.analytics import DataLeakageAuditor

leakage_auditor = DataLeakageAuditor(target_column=settings.target_column)
leakage_auditor.row_overlap_report(
    raw_train_df,
    raw_test_df,
    holdout_name="raw_test",
)

,check,reference,holdout,train_rows,holdout_rows,overlap_count,overlap_rate,passed
0,exact_row_overlap,train,raw_test,40500,4500,0,0.0,True


## 4. Train-only dataset inspection

In [5]:
from credit_risk_lab.infrastructure.analytics import DatasetInspector

inspector = DatasetInspector(raw_train_df)
summary = inspector.summary()
summary

DatasetSummary(rows=40500, columns=14, duplicate_rows=0, memory_mb=13.4709)

## 5. Train-only column summary

In [6]:
column_summary = inspector.column_summary(sample_size=2)
column_summary

,column,dtype,missing,missing_rate,cardinality,total_rows,all_values_unique,examples
0,person_age,float64,0,0.0,59,40500,False,"[24.0, 28.0]"
1,person_gender,object,0,0.0,2,40500,False,"[male, female]"
2,person_education,object,0,0.0,5,40500,False,"[Associate, Associate]"
3,person_income,float64,0,0.0,31335,40500,False,"[161487.0, 130324.0]"
4,person_emp_exp,int64,0,0.0,63,40500,False,"[1, 5]"
5,person_home_ownership,object,0,0.0,4,40500,False,"[RENT, MORTGAGE]"
6,loan_amnt,float64,0,0.0,4124,40500,False,"[25000.0, 27400.0]"
7,loan_intent,object,0,0.0,6,40500,False,"[DEBTCONSOLIDATION, EDUCATION]"
8,loan_int_rate,float64,0,0.0,1292,40500,False,"[10.37, 10.99]"
9,loan_percent_income,float64,0,0.0,64,40500,False,"[0.15, 0.21]"


## 6. Train-only target distribution

In [7]:
target_distribution = inspector.target_distribution(settings.target_column)
display(target_distribution)

from credit_risk_lab.infrastructure.visualization import plot_target_distribution

plot_target_distribution(target_distribution).show()

,class,rows,rate
0,0,31500,0.777778
1,1,9000,0.222222
